In [0]:
from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"

valid_status = ["paid", "pending", "rejected"]
valid_methods = ["SEPA", "bank_transfer", "card"]

In [0]:
payments_bronze = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_payments")

claims = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims") \
    .select("claim_id", "claim_date") \
    .dropDuplicates()

In [0]:
payments_prepared = (
    payments_bronze
    .withColumn("payment_id", F.trim(F.col("payment_id")))
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("payment_status", F.lower(F.trim(F.col("payment_status"))))
    .withColumn("payment_method", F.lower(F.trim(F.col("payment_method"))))
    .withColumn("payment_amount", F.col("payment_amount").cast("double"))
    .withColumn("payment_date", F.to_date(F.col("payment_date")))
)

In [0]:
payments_joined = (
    payments_prepared.alias("p")
    .join(claims.alias("c"), "claim_id", "left")
    .withColumn("claim_exists", F.col("c.claim_id").isNotNull())
    .withColumn("claim_date_ref", F.col("c.claim_date"))
)

In [0]:
invalid_payments = (
    payments_joined
    .filter(
        F.col("payment_id").isNull()
        | F.col("claim_id").isNull()
        | (~F.col("claim_exists"))
        | (~F.col("payment_status").isin(valid_status))
        | (~F.col("payment_method").isin(valid_methods))
        | (F.col("payment_amount") < 0)
        | (F.col("payment_date") < F.col("claim_date_ref"))
    )
    .withColumn("record_id", F.col("payment_id"))
    .withColumn("source_table", F.lit("bronze_payments"))
    .withColumn("error_reason", F.lit("payment_validation_failed"))
    .withColumn("error_severity", F.lit("HIGH"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn(
        "original_record_json",
        F.to_json(F.struct(*[F.col(c) for c in payments_prepared.columns]))
    )
)

In [0]:
silver_payments = (
    payments_joined
    .filter(
        F.col("payment_id").isNotNull()
        & F.col("claim_id").isNotNull()
        & F.col("claim_exists")
        & F.col("payment_status").isin(valid_status)
        & F.col("payment_method").isin(valid_methods)
        & (F.col("payment_amount") >= 0)
        & (F.col("payment_date") >= F.col("claim_date_ref"))
    )
    .drop("claim_exists", "claim_date_ref")
    .dropDuplicates(["payment_id"])
)

In [0]:
silver_payments.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_payments")

In [0]:
(
    invalid_payments
    .select(
        "record_id",
        "source_table",
        "error_reason",
        "error_severity",
        "quarantine_timestamp",
        "source_file_name",
        "ingest_run_id",
        "original_record_json"
    )
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_payments")
)

In [0]:
print("Bronze payments:", payments_bronze.count())
print("Silver payments:", spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_payments").count())
print("Quarantine payments:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_payments").count())